In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import yaml
from collections import Counter

In [7]:

# repo root = pasta pai de notebooks/
REPO_ROOT = Path.cwd().parent

KAGGLE_ROOT = REPO_ROOT / "data" / "kaggle_raw" / "src" / "dataset" / "dataset_augmented"
MAPPING_FILE = REPO_ROOT / "data" / "class_mapping.yaml"

print("REPO_ROOT:", REPO_ROOT)
print("KAGGLE_ROOT exists:", KAGGLE_ROOT.exists())
print("MAPPING_FILE exists:", MAPPING_FILE.exists())

xml_files = list(KAGGLE_ROOT.rglob("*.xml"))
print("Total XML files found:", len(xml_files))

REPO_ROOT: /Users/vilella/Documents/fiap/pos_ia/iadt_fase05_final
KAGGLE_ROOT exists: True
MAPPING_FILE exists: True
Total XML files found: 8700


In [8]:
all_classes = []

for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for obj in root.findall("object"):
        all_classes.append(obj.findtext("name"))

unique_classes = sorted(set(all_classes))
print("Total unique original classes:", len(unique_classes))
unique_classes[:20]

Total unique original classes: 111


['api',
 'aws_amazon_api_gateway',
 'aws_amazon_cloudfront',
 'aws_amazon_cloudwatch',
 'aws_amazon_dynamodb',
 'aws_amazon_ec2',
 'aws_amazon_ec2_auto_scaling',
 'aws_amazon_elastic_block_store',
 'aws_amazon_elastic_container_service',
 'aws_amazon_elastic_kubernetes_service',
 'aws_amazon_elasticache',
 'aws_amazon_rds',
 'aws_amazon_redshift',
 'aws_amazon_route_53',
 'aws_amazon_simple_notification_service',
 'aws_amazon_simple_queue_service',
 'aws_amazon_simple_storage_service',
 'aws_amazon_virtual_private_cloud',
 'aws_application_load_balancer',
 'aws_aurora_amazon_rds_instance']

In [9]:
len(unique_classes)

111

In [10]:
with open(MAPPING_FILE, "r") as f:
    mapping = yaml.safe_load(f)

def map_class(original_name):
    original_name = original_name.lower()
    for target_class, keywords in mapping.items():
        for kw in keywords:
            if kw in original_name:
                return target_class
    return None

mapped = [map_class(c) for c in unique_classes]
mapping_df = pd.DataFrame({
    "original_class": unique_classes,
    "mapped_class": mapped
})

mapping_df.head(20)

,original_class,mapped_class
0,api,NaN
1,aws_amazon_api_gateway,gateway
2,aws_amazon_cloudfront,NaN
3,aws_amazon_cloudwatch,NaN
4,aws_amazon_dynamodb,database
5,aws_amazon_ec2,NaN
6,aws_amazon_ec2_auto_scaling,NaN
7,aws_amazon_elastic_block_store,NaN
8,aws_amazon_elastic_container_service,service
9,aws_amazon_elastic_kubernetes_service,service


In [11]:
coverage = mapping_df["mapped_class"].value_counts(dropna=False)
coverage

mapped_class
NaN                 59
service             24
database            18
gateway              5
web_app              3
external_service     1
user                 1
Name: count, dtype: int64

In [13]:
valid_images = []

for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    keep = False
    
    for obj in root.findall("object"):
        original = obj.findtext("name")
        if map_class(original) is not None:
            keep = True
            break
    
    if keep:
        valid_images.append(xml_file)

print("Images containing at least one mapped class:", len(valid_images))

Images containing at least one mapped class: 8690
